# Stage1 现金与上限联合 SOXL 局部净价值

冻结2018–2022原架构OOF训练账本，比较SOXL与既有SOXX动作在相同0.10 beta包络下的09:47至下一日09:47含Tiered费局部价值。仅训练标签，不拟合或运行连续策略账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import copy,hashlib,json,sys,shlex,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1]
sys.path.insert(0,str(root))
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
card=root/'research/stage1_joint_cash_cap_soxl_0bp_local_value_20261024'
plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for path,key in [(root/src['prior_joint_plan'],'prior_joint_plan_sha256'),(root/'research/stage1_joint_cash_cap_soxx_0bp_local_value_20261023/summary.json','prior_joint_summary_sha256'),(root/src['training_reference_card']/'summary.json','training_reference_summary_sha256'),(root/src['fee_plan'],'fee_plan_sha256'),(root/'src/backtest/ibkr_tiered.py','fee_module_sha256'),(root/src['provider']/'qlib_store_manifest.json','provider_manifest_sha256')]:
 assert sha(path)==src[key],(path,key)
source_summary=json.loads((root/src['training_reference_card']/'summary.json').read_text())
assert source_summary['recorder_id']==src['training_reference_recorder_id'] and source_summary['experiment_id']==src['training_reference_experiment_id'] and source_summary['nas_readback']['verified']
prior_card=root/'research/stage1_joint_cash_cap_soxx_0bp_local_value_20261023'
prior_summary=json.loads((prior_card/'summary.json').read_text())
assert prior_summary['nas_readback']['verified'] and prior_summary['nas_readback']['sha256']==prior_summary['nas_evidence']['sha256']
fee_plan=json.loads((root/src['fee_plan']).read_text());provider=(root/src['provider']).resolve();tracking=(root/src['training_reference_card']/'qlib_recorder').resolve()
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':Path(src['training_reference_card']).name}})
ref=R.get_recorder(experiment_name=Path(src['training_reference_card']).name,recorder_id=src['training_reference_recorder_id'])
frame=ref.load_object('inputs/source_frame').reset_index(drop=True)
bundle=ref.load_object('paths/reference_0');native=ref.load_object('native/reference_0');action=ref.load_object('inputs/action_plan')
path=bundle['replay'].reset_index(drop=True);targets=bundle['targets'].reset_index(drop=True)
assert native['audit']['status']=='passed' and native['audit']['slippage_bps']==0 and len(frame)==len(path)==len(targets)==1061
assert frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(targets.trade_date)
cal=list(pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date)
assert cal==list(pd.to_datetime(frame.trade_date).dt.date)
assert action['recovery_spec']['minimum_cash_weight']==.15
assert (pd.to_datetime(frame.feature_available_at,utc=True)<=pd.to_datetime(frame.decision_at,utc=True)).all()
with R.uri_context(str((prior_card/'qlib_recorder').resolve())):
 prior_rec=R.get_recorder(experiment_id=str(prior_summary['experiment_id']),recorder_id=prior_summary['recorder_id'])
 prior_labels=prior_rec.load_object('analysis/labels')
assert len(prior_labels)==1061 and pd.to_datetime(prior_labels.trade_date).equals(pd.to_datetime(frame.trade_date))
state=card/'run_state.json';state.write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'label_passes_started':1},indent=2)+'\n')
fees=TieredFees(fee_plan['tariff'],slippage_bps=0);q=np.zeros(2);cash=100000.;records=[];max_ledger_error=0.;max_branch_error=0.;whole_share_changed=0
for i,row in frame.iterrows():
 saved=path.iloc[i];target=targets.iloc[i][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
 raw=row[['soxl_weight','soxx_weight']].to_numpy(float);cap=float(saved.current_beta_cap)
 raw_beta=float(3*raw[0]+raw[1]);base_beta=float(3*target[0]+target[1]);binding=bool(raw_beta>cap+1e-9)
 ceiling=cap+(.10 if binding else 0.)
 delta_beta=max(0.,min(.10,3*max(0.,target[2]-.15),max(0.,ceiling-base_beta))) if bool(saved.execute_open_target) else 0.
 delta_weight=delta_beta/3
 eligible=bool(saved.execute_open_target and delta_weight>1e-10);mature=eligible and i+1<len(frame)
 candidate_target=target+np.array([delta_weight,0.,-delta_weight])
 assert candidate_target.min()>=-1e-12 and abs(candidate_target.sum()-1)<1e-9 and candidate_target[2]>=.15-1e-10 and 3*candidate_target[0]+candidate_target[1]<=ceiling+1e-9
 today=row[['soxl_open','soxx_open']].to_numpy(float);raw_today=row[['raw_soxl_open','raw_soxx_open']].to_numpy(float)
 stamp=pd.Timestamp(row.trade_date)+pd.Timedelta(hours=9,minutes=47);nav=float(q@today+cash)
 label=np.nan;available=pd.NaT
 if mature:
  next_row=frame.iloc[i+1];reset=targets.iloc[i+1][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
  fee_a=copy.deepcopy(fees);fee_b=copy.deepcopy(fees);assert fee_a.monthly_shares==fee_b.monthly_shares==fees.monthly_shares
  a=settle_tiered_event(q,cash,desired=target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fee_a)
  b=settle_tiered_event(q,cash,desired=candidate_target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fee_b)
  assert np.isclose(a['nav_before'],b['nav_before'],rtol=0,atol=1e-8) and b['cash']>=-1e-7
  whole_share_changed+=int(np.max(np.abs(a['quantities']-b['quantities'])*today)/nav>1e-8)
  tomorrow=next_row[['soxl_open','soxx_open']].to_numpy(float);raw_tomorrow=next_row[['raw_soxl_open','raw_soxx_open']].to_numpy(float)
  nextstamp=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=47)
  ar=settle_tiered_event(a['quantities'],a['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=fee_a)
  br=settle_tiered_event(b['quantities'],b['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=fee_b)
  av=float(ar['quantities']@tomorrow+ar['cash']);bv=float(br['quantities']@tomorrow+br['cash']);assert np.isfinite([av,bv]).all() and min(av,bv)>0
  label=(bv-av)/nav;available=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=49)
 control=settle_tiered_event(q,cash,desired=target if bool(saved.execute_open_target) else None,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fees)
 if mature:
  err=float(np.max(np.abs(np.r_[a['quantities'],a['cash']]-np.r_[control['quantities'],control['cash']])));max_branch_error=max(max_branch_error,err);assert err<=1e-8
 err=float(np.max(np.abs(np.r_[control['quantities'],control['cash']]-saved[['ending_soxl_quantity','ending_soxx_quantity','ending_cash']].to_numpy(float))))
 max_ledger_error=max(max_ledger_error,err);assert err<=1e-6,(i,err)
 records.append({'signal_date':row.signal_date,'trade_date':row.trade_date,'decision_at':row.decision_at,'eligible':eligible,'cap_binding':binding,'delta_target_beta':delta_beta,'delta_SOXL_weight':delta_weight,'label_available_at':available,'soxl_local_fee_net_advantage':label})
 q,cash=control['quantities'],control['cash']
labels=pd.DataFrame(records);valid=labels[labels.soxl_local_fee_net_advantage.notna()]
assert len(labels)==1061 and len(valid)>0 and (pd.to_datetime(valid.label_available_at)>pd.to_datetime(valid.decision_at)).all()
annual=[];quarter=[]
for year,g in labels.groupby(pd.to_datetime(labels.trade_date).dt.year):
 z=g.soxl_local_fee_net_advantage.dropna()
 annual.append({'year':int(year),'sessions':len(g),'eligible':int(g.eligible.sum()),'mature':len(z),'binding_mature':int((g.cap_binding & g.soxl_local_fee_net_advantage.notna()).sum()),'nonbinding_mature':int((~g.cap_binding & g.soxl_local_fee_net_advantage.notna()).sum()),'positive':int((z>0).sum()),'negative':int((z<0).sum()),'mean':float(z.mean()) if len(z) else None})
for (year,qt),g in labels.groupby([pd.to_datetime(labels.trade_date).dt.year,pd.to_datetime(labels.trade_date).dt.quarter]):
 z=g.soxl_local_fee_net_advantage.dropna()
 quarter.append({'year':int(year),'quarter':int(qt),'sessions':len(g),'mature':len(z),'mean':float(z.mean()) if len(z) else None})
by_layer={}
for name,g in [('binding',valid[valid.cap_binding]),('nonbinding',valid[~valid.cap_binding])]:
 by_layer[name]={'mature':len(g),'mean':float(g.soxl_local_fee_net_advantage.mean()) if len(g) else None,'positive':int((g.soxl_local_fee_net_advantage>0).sum()),'negative':int((g.soxl_local_fee_net_advantage<0).sum())}
match=labels.soxl_local_fee_net_advantage.notna() & prior_labels.joint_local_fee_net_advantage.notna()
prior_matched=prior_labels.loc[match,'joint_local_fee_net_advantage'].to_numpy(float)
soxl_matched=labels.loc[match,'soxl_local_fee_net_advantage'].to_numpy(float)
comparison={'matched_mature':int(match.sum()),'SOXL_mean':float(soxl_matched.mean()),'SOXX_mean':float(prior_matched.mean()),'SOXL_minus_SOXX_mean':float((soxl_matched-prior_matched).mean()),'correlation':float(np.corrcoef(soxl_matched,prior_matched)[0,1])}
result={'decision':'training_only_SOXL_cash_cap_local_value_no_fit_or_account','plan_sha256':sha(card/'plan.json'),'sessions':len(labels),'eligible':int(labels.eligible.sum()),'mature':len(valid),'whole_share_changed_mature':whole_share_changed,
        'mean_local_advantage':float(valid.soxl_local_fee_net_advantage.mean()),'median_local_advantage':float(valid.soxl_local_fee_net_advantage.median()),'positive':int((valid.soxl_local_fee_net_advantage>0).sum()),'negative':int((valid.soxl_local_fee_net_advantage<0).sum()),
        'annual':annual,'quarter':quarter,'by_layer':by_layer,'prior_SOXX_matched_comparison':comparison,
        'max_control_cash_share_ledger_error_usd':max_ledger_error,'max_branch_control_identity_error_usd':max_branch_error,'last_label_unknown':str(labels.trade_date.iloc[-1].date()),
        'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'label_passes':1,'historical_independence':False,
        'interpretation':'2018-2022 original-architecture OOF local 0bp Tiered target only. Same-next-target reset is not a continuous f696 Stage1 own-state account or independent forward evidence.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan);rec.save_objects(artifact_path='analysis',labels=labels,annual=annual,quarter=quarter,result=result)
 rec.log_metrics(mature_rows=len(valid),binding_rows=by_layer['binding']['mature'],nonbinding_rows=by_layer['nonbinding']['mature'],mean_local_advantage=result['mean_local_advantage'],new_model_fits=0,new_native_accounts=0)
 rid=rec.id;eid=str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'source_recorder_id':src['training_reference_recorder_id'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/labels','analysis/annual','analysis/quarter','analysis/result']}}
binding=json.loads((root/src['training_reference_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30')
(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='soxl-cap-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored)
 rel=Path('qlib_recorder')/eid/rid/'artifacts/analysis/labels';assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote,'scoped_restored_labels_sha256':sha(card/rel)}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
state.write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'label_passes_completed':1,'recorder_id':rid},indent=2)+'\n')
print(json.dumps({'result':result,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False))


{"result": {"decision": "training_only_SOXL_cash_cap_local_value_no_fit_or_account", "plan_sha256": "a7f51a157f98192edc9dfec832c0e34231d0fdc58c10ffd8bc826c2d61b6d2a0", "sessions": 1061, "eligible": 689, "mature": 688, "whole_share_changed_mature": 657, "mean_local_advantage": 4.957468748976795e-05, "median_local_advantage": 3.934676010001357e-06, "positive": 345, "negative": 311, "annual": [{"year": 2018, "sessions": 53, "eligible": 19, "mature": 19, "binding_mature": 18, "nonbinding_mature": 1, "positive": 11, "negative": 8, "mean": 7.445166997873025e-05}, {"year": 2019, "sessions": 252, "eligible": 138, "mature": 138, "binding_mature": 39, "nonbinding_mature": 99, "positive": 63, "negative": 56, "mean": 2.9222363680665355e-05}, {"year": 2020, "sessions": 253, "eligible": 202, "mature": 202, "binding_mature": 182, "nonbinding_mature": 20, "positive": 118, "negative": 83, "mean": 0.00019662442935676125}, {"year": 2021, "sessions": 252, "eligible": 82, "mature": 82, "binding_mature": 56

## 中文结论

# 现金与上限联合 SOXL 动作：训练期局部价值跨年失稳

按冻结计划仅在原架构2018–2022 OOF 训练账本生成一次 0bp 附加滑点、IBKR Tiered费用后的局部标签。与既有现金转SOXX标签使用相同最多 **0.10 beta** 包络，候选把至多 `delta_beta/3` 的现金权重给SOXL；克隆原09:46资金、整股持仓与月度费用计数，在09:47执行，两支于次日09:47共同目标复位。原控制账本资金/持仓误差 **0**，分支控制误差 **0**。1061个Qlib交易日中689日符合执行条件、688标签成熟、657日改变整股持仓；全年、季度、受限/未受限层均保留在Recorder。

688日SOXL分支平均局部净优势 **+0.00004957**（约0.50bp），受限540日 **+0.00005148**，未受限148日 **+0.00004264**。2020/2021完整年的均值 **+0.00019662/+0.00035850**，2022完整年 **−0.00016378**（约−1.64bp）。2022年未受限层只有 **2** 个成熟标签，无法检验该层跨年稳定性。与同日同beta包络的旧SOXX局部标签在688个匹配日比较，SOXL均值 **0.50bp**，SOXX **0.84bp**，SOXL减SOXX **−0.35bp**；相关系数 **0.99494**。这更像同一市场暴露的强弱差，而非已证实的独立SOXL机会。

按计划的跨年符号和层内支持停止条件，决定 `not_promoted_fixed_SOXL_joint_local_value_2022_negative_and_nonbinding_support_2`：**不拟合接收器、不运行2023+完整候选账户**，不根据2020/2021好年份或个别季度调动作、beta尺寸、现金底线。局部共同复位标签不代表连续自身风险状态策略的收益；训练OOF账本也不是精确f696评价期父本。没有超过SOXX的净值、2/5bp或独立前向证据。下一机制需要能在决策时区分2022式现金保护与2020/2021式现金机会，并证明在完整f696配置及自身风险容量内实际形成动作。

Qlib Recorder `9f4753640e9247a18adff4cab4bb57f8` 已记录全部标签、年度、季度和摘要；NAS归档SHA-256 `15f713a01daa410cafae50955e4f6abab5f0baab6fd70a246b00745755ccc5e1` 定向恢复标签哈希通过。